In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

In [ ]:
os.makedirs("../../result/HEK293T_HCT116_permute/ranking/", exist_ok=True)

### Inputs: technical covariates of every real vector, the permutation, coverage class

In [ ]:
def load_input(infile):
    return pd.read_csv(infile, sep="\t", index_col=0)


def node_degree(infile):
    ppi = pd.read_csv(infile, sep="\t")
    return pd.concat([ppi["GeneA"], ppi["GeneB"]]).value_counts()


def knn_mean_cosine(unit, k, chunk=2000):
    """
    Mean cosine of every row to its k nearest rows, self excluded.
    """
    unit = unit.astype(np.float32)
    n = unit.shape[0]
    out = np.empty(n, dtype=np.float64)
    for start in range(0, n, chunk):
        stop = min(start + chunk, n)
        sim = unit[start:stop] @ unit.T
        sim[np.arange(stop - start), np.arange(start, stop)] = -np.inf
        top = np.partition(sim, n - k, axis=1)[:, n - k:]
        out[start:stop] = top.mean(axis=1)
    return out


def file_covariates(emb, infile):
    """
    One row per protein of one input embedding: degree, norm, hubness.
    """
    values = emb.values.astype(np.float64)
    norm = np.linalg.norm(values, axis=1)
    unit = values / np.maximum(norm, 1e-12)[:, None]
    centroid = values.mean(axis=0)
    centroid = centroid / max(np.linalg.norm(centroid), 1e-12)
    frame = pd.DataFrame(index=emb.index)
    degree = node_degree(infile).reindex(emb.index).fillna(0)
    frame["degree"] = degree.values
    frame["log_degree"] = np.log1p(frame["degree"])
    frame["norm"] = norm
    frame["cos_centroid"] = unit @ centroid
    frame["knn_cos"] = knn_mean_cosine(unit, 10)
    return frame


def recover_permutation(real, perm):
    """
    if permuted name A carries the original vector of B, it records {A:B}
    """
    lookup = {}
    for name, row in zip(real.index, np.ascontiguousarray(real.values)):
        lookup.setdefault(row.tobytes(), []).append(name)
    mapping = {}
    for name, row in zip(perm.index, np.ascontiguousarray(perm.values)):
        hits = lookup.get(row.tobytes())
        if hits is not None and len(hits) == 1:
            mapping[name] = hits[0]
    return pd.Series(mapping)


covariates = {}
permute_map = {}
real_index = {}
for file, infile_real, infile_perm, infile_ppi in [
    (
        "BioPlex-HEK293T",
        "../../result/HEK293T_HCT116/input/BioPlex-HEK293T.tsv",
        "../../result/HEK293T_HCT116_permute/input/BioPlex-HEK293T.tsv",
        "../../result/HEK293T_HCT116/input/BioPlex-HEK293T.ppi"
    ),
    (
        "SECMX-HEK293T",
        "../../result/HEK293T_HCT116/input/SECMX-HEK293T.tsv",
        "../../result/HEK293T_HCT116_permute/input/SECMX-HEK293T.tsv",
        "../../result/HEK293T_HCT116/input/SECMX-HEK293T.ppi"
    ),
    (
        "BioPlex-HCT116",
        "../../result/HEK293T_HCT116/input/BioPlex-HCT116.tsv",
        "../../result/HEK293T_HCT116_permute/input/BioPlex-HCT116.tsv",
        "../../result/HEK293T_HCT116/input/BioPlex-HCT116.ppi"
    ),
    (
        "SECMX-HCT116",
        "../../result/HEK293T_HCT116/input/SECMX-HCT116.tsv",
        "../../result/HEK293T_HCT116_permute/input/SECMX-HCT116.tsv",
        "../../result/HEK293T_HCT116/input/SECMX-HCT116.ppi"
    ),
]:
    real = load_input(infile_real)
    perm = load_input(infile_perm)
    assert set(real.index) == set(perm.index), file
    real_index[file] = real.index
    covariates[file] = file_covariates(real, infile_ppi)
    permute_map[file] = recover_permutation(real, perm)
    
identity = {
    file: pd.Series(cov.index, index=cov.index)
    for file, cov in covariates.items()
}

In [ ]:
# coverage class of every query protein with a base counterpart
query_names = pd.Index(sorted(
    set(real_index["BioPlex-HEK293T"])
    | set(real_index["SECMX-HEK293T"])
))
base_names = (
    set(real_index["BioPlex-HCT116"])
    | set(real_index["SECMX-HCT116"])
)
coverage = pd.DataFrame({
    "BioPlex": query_names.isin(real_index["BioPlex-HEK293T"]),
    "SECMX": query_names.isin(real_index["SECMX-HEK293T"]),
    "base": query_names.isin(base_names),
}, index=query_names).astype(int)
cls_all = coverage.astype(str).agg("".join, axis=1)
names = cls_all.index[coverage["base"] == 1]
cls = cls_all.reindex(names)

### Scores of the real and the permuted run

In [ ]:
def load_score(infile):
    if infile.endswith(".cca"):
        frame = pd.read_csv(infile, sep="\t")
        return frame.set_index("gene")["score"].astype(float)
    frame = pd.read_csv(infile, sep="\t", index_col=0)
    return frame["specific_score"].astype(float)

### Technical model on the permuted run, removed from the real score within coverage class

In [ ]:
def design_matrix(index, files, carrier):
    """
    construct feature.
    carrier[file] maps a name to the real protein whose vector it carries:
    the identity for the real run, the recovered permutation for the permuted run.
    """
    blocks = []
    for file, slot in files:
        carried = carrier[file].reindex(index)
        block = covariates[file].reindex(carried.values)[
            ["log_degree", "norm", "cos_centroid", "knn_cos"]
        ]
        block.index = index
        block.columns = [f"{slot}_{f}" for f in block.columns]
        blocks.append(block)
    return pd.concat(blocks, axis=1)


def technical_expectation(perm_score):
    """
    OLS of the permuted score on the covariates
    """
    tech = pd.Series(np.nan, index=names, dtype=float)
    r2 = {}
    for c, files in {
        "111": [
            ("BioPlex-HEK293T", "qB"),
            ("SECMX-HEK293T", "qS"),
            ("BioPlex-HCT116", "bB"),
            ("SECMX-HCT116", "bS")
        ],
        "101": [("BioPlex-HEK293T", "qB"), ("BioPlex-HCT116", "bB")],
        "011": [("SECMX-HEK293T", "qS"), ("SECMX-HCT116", "bS")]
    }.items():
        perm_cls = cls_all.reindex(perm_score.index).values
        perm_names = perm_score.index[perm_cls == c]
        X_perm = design_matrix(perm_names, files, permute_map)
        y = perm_score.reindex(perm_names)
        keep = X_perm.notna().all(axis=1) & y.notna()
        model = LinearRegression().fit(
            X_perm[keep].values,
            y[keep].values,
        )
        fitted = model.predict(X_perm[keep].values)
        ss_res = ((y[keep] - fitted) ** 2).sum()
        ss_tot = ((y[keep] - y[keep].mean()) ** 2).sum()
        r2[c] = 1 - ss_res / ss_tot

        real_names = names[cls.values == c]
        X_real = design_matrix(real_names, files, identity)
        ok = X_real.notna().all(axis=1)
        pred = model.predict(X_real[ok].values)
        tech.loc[real_names[ok.values]] = pred
    return tech, r2


def remove_technical(real, tech):
    """
    The permute-removed score:
    real score minus its OLS fit on the technical expectation.
    """
    out = pd.Series(np.nan, index=real.index)
    for c in cls.unique():
        mask = (cls == c) & real.notna() & tech.notna()
        if mask.sum() < 20 or tech[mask].std() < 1e-12:
            out[mask] = real[mask] - real[mask].mean()
            continue
        slope, intercept = np.polyfit(
            tech[mask].values,
            real[mask].values,
            1,
        )
        out[mask] = real[mask] - (intercept + slope * tech[mask])
    return out

### Output: one ranking per group

In [ ]:
for infile_real, infile_perm, outfile in [
    (
        "../../result/HEK293T_HCT116/cca/FromInputConcat_HEK293T.cca",
        "../../result/HEK293T_HCT116_permute/cca/FromInputConcat_HEK293T.cca",
        "../../result/HEK293T_HCT116_permute/ranking/FromInputConcat_HEK293T.ranking"
    ),
    (
        "../../result/HEK293T_HCT116/cca/AEconcat_HEK293T.cca",
        "../../result/HEK293T_HCT116_permute/cca/AEconcat_HEK293T.cca",
        "../../result/HEK293T_HCT116_permute/ranking/AEconcat_HEK293T.ranking"
    ),
    (
        "../../result/HEK293T_HCT116/cca/GPSseparate_HEK293T.cca",
        "../../result/HEK293T_HCT116_permute/cca/GPSseparate_HEK293T.cca",
        "../../result/HEK293T_HCT116_permute/ranking/GPSseparate_HEK293T.ranking"
    ),
    (
        "../../result/HEK293T_HCT116/cca/GPScombine_HEK293T.cca",
        "../../result/HEK293T_HCT116_permute/cca/GPScombine_HEK293T.cca",
        "../../result/HEK293T_HCT116_permute/ranking/GPScombine_HEK293T.ranking"
    ),
    (
        "../../result/HEK293T_HCT116/cca/promergesubtract_HCT116anchor_HEK293T.cca",
        "../../result/HEK293T_HCT116_permute/cca/promergesubtract_HCT116anchor_HEK293T.cca",
        "../../result/HEK293T_HCT116_permute/ranking/promergesubtract_HCT116anchor_HEK293T.ranking"
    ),
    (
        "../../result/HEK293T_HCT116/cca/promergeclub_HCT116anchor_HEK293T.cca",
        "../../result/HEK293T_HCT116_permute/cca/promergeclub_HCT116anchor_HEK293T.cca",
        "../../result/HEK293T_HCT116_permute/ranking/promergeclub_HCT116anchor_HEK293T.ranking"
    ),
    (
        "../../result/HEK293T_HCT116/coemb/promergeclub_HCT116anchor/HEK293T/HEK293T_specific_score.tsv",
        "../../result/HEK293T_HCT116_permute/coemb/promergeclub_HCT116anchor/HEK293T/HEK293T_specific_score.tsv",
        "../../result/HEK293T_HCT116_permute/ranking/promergeclub_HCT116anchor_HEK293T_specific_score.ranking"
    ),
]:
    real = load_score(infile_real).reindex(names)
    perm = load_score(infile_perm)
    tech, r2 = technical_expectation(perm)
    score = remove_technical(real, tech)

    out = pd.DataFrame({
        "gene": names,
        "score": score.values,
        "rank": score.rank(ascending=False, method="min").values,
        "rank_in_class": score.groupby(cls).rank(
            ascending=False, method="min"
        ).values,
        "coverage_class": cls.values,
        "real_score": real.values,
        "technical_expectation": tech.values,
    }).dropna(subset=["score"]).sort_values("rank")
    out["rank"] = out["rank"].astype(int)
    out["rank_in_class"] = out["rank_in_class"].astype(int)
    out.to_csv(outfile, sep="\t", index=False)